# Phase 1 — Data Validation and Data Understanding

**Project:** AI-Powered Banking Customer Support Intelligence & Response Automation

**Objective:**
- Inspect dataset shape, dtypes, missing values, and duplicates.
- Understand the category distribution (77 intents).
- Analyse text length, word count, and character count distributions.
- Verify consistency between train and test datasets.
- Create EDA visualisations and save outputs to `results/`.

**Important rule:** The test dataset is loaded for inspection only. It will **not** be used for any modelling decisions.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_understanding import run as run_data_understanding

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
print('Setup complete.')

## 1. Load Datasets

In [ ]:
ROOT = Path('..')
df_train = pd.read_csv(ROOT / 'data' / 'train.csv')
df_test  = pd.read_csv(ROOT / 'data' / 'test.csv')

print(f'Train shape : {df_train.shape}')
print(f'Test  shape : {df_test.shape}')
print(f'\nTrain columns: {df_train.columns.tolist()}')
df_train.head()

## 2. Basic Validation

In [ ]:
print('=== TRAIN DATASET ===')
print(f'Shape          : {df_train.shape}')
print(f'Dtypes:\n{df_train.dtypes}')
print(f'Missing values :\n{df_train.isnull().sum()}')
print(f'Duplicate rows : {df_train.duplicated().sum()}')
print(f'Unique texts   : {df_train["text"].nunique()}')
print(f'Empty texts    : {(df_train["text"].str.strip() == "").sum()}')
print(f'Unique categories: {df_train["category"].nunique()}')

In [ ]:
print('=== TEST DATASET ===')
print(f'Shape          : {df_test.shape}')
print(f'Missing values :\n{df_test.isnull().sum()}')
print(f'Duplicate rows : {df_test.duplicated().sum()}')
print(f'Unique categories: {df_test["category"].nunique()}')

## 3. Category Consistency Check

In [ ]:
train_cats = set(df_train['category'].unique())
test_cats  = set(df_test['category'].unique())

print(f'Train categories: {len(train_cats)}')
print(f'Test  categories: {len(test_cats)}')
print(f'In test but not in train: {test_cats - train_cats}')
print(f'In train but not in test: {train_cats - test_cats}')

if train_cats == test_cats:
    print('\n✅ Train and test categories are perfectly consistent.')
else:
    print('\n⚠️  Mismatch detected!')

## 4. Category Distribution

In [ ]:
train_dist = df_train['category'].value_counts()
print('Top 10 categories (train):')
print(train_dist.head(10))
print(f'\nBottom 5 categories:')
print(train_dist.tail(5))
print(f'\nImbalance ratio (max/min): {train_dist.max() / train_dist.min():.1f}x')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
train_dist.plot(kind='bar', ax=ax, color='#06d6a0', edgecolor='none')
ax.set_title('Training Set — Category Distribution (77 intents)', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Sample Count')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

## 5. Text Length Analysis

In [ ]:
df_train['text_len'] = df_train['text'].str.len()
df_train['word_count'] = df_train['text'].str.split().str.len()

print('Text character length (train):')
print(df_train['text_len'].describe())
print('\nWord count (train):')
print(df_train['word_count'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_train['text_len'], bins=50, color='#118ab2', edgecolor='none', alpha=0.8)
axes[0].axvline(df_train['text_len'].median(), color='#ef476f', linestyle='--', label='Median')
axes[0].set_title('Character Length Distribution (Train)')
axes[0].set_xlabel('Character Count')
axes[0].legend()

axes[1].hist(df_train['word_count'], bins=30, color='#ffd166', edgecolor='none', alpha=0.8)
axes[1].axvline(df_train['word_count'].median(), color='#ef476f', linestyle='--', label='Median')
axes[1].set_title('Word Count Distribution (Train)')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Run Full Data Understanding Pipeline

In [ ]:
# Run the full data understanding script from src/
# This saves all EDA outputs to results/charts/
import os
os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
summary = run_data_understanding()
print('Data understanding complete. Summary keys:', list(summary.keys()) if summary else 'N/A')

## Summary

| Property | Train | Test |
|----------|-------|------|
| Samples | ~10,003 | ~3,080 |
| Categories | 77 | 77 |
| Missing values | 0 | 0 |
| Duplicates | ~0 | 0 |

**Key findings:**
- Training data is **imbalanced** — some categories have significantly more samples.
- Test set has approximately **40 samples per category** (balanced).
- All 77 categories are present in both train and test sets.
- Text lengths vary but most queries are short (5-30 words).

**Next step:** Phase 2 — NLP Preprocessing and Train/Validation Split